# 🏠 WebScraperInmobiliario — Colima

Scraper inmobiliario para los 10 municipios de Colima.  
Portales: **Inmuebles24 · Lamudi · Vivanuncios · MercadoLibre**

---
### Flujo de trabajo
1. ⚙️ Instalar dependencias y montar Google Drive
2. 🗄️ Inicializar la base de datos
3. 🕷️ Ejecutar el scraping
4. 📊 Explorar los datos
5. 📈 Análisis de mercado
6. 📥 Exportar resultados

> **Ejecuta las celdas en orden la primera vez.**  
> En sesiones posteriores puedes saltar la Sección 0 si Drive ya está montado.

---
## Sección 0 — Configuración del entorno
*(Solo necesitas correr esta sección una vez por sesión de Colab)*

In [ ]:
# ── 0.1  Clonar repositorio e instalar dependencias ──────────────────────────
import os, subprocess, sys

REPO_URL = "https://github.com/Kesarleon/WebScraperInmobiliario.git"  # ← ajusta si cambia
REPO_DIR = "/content/WebScraperInmobiliario"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("✅ Repositorio clonado.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
    print("✅ Repositorio actualizado.")

# Agrega el repo al path de Python
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Instala dependencias
%pip install -q -r {REPO_DIR}/requirements.txt
print("✅ Dependencias instaladas.")

In [ ]:
# ── 0.2  Montar Google Drive (para persistir la BD entre sesiones) ────────────
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_DIR = "/content/drive/MyDrive/InmobiliarioColima"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"✅ Carpeta en Drive: {DRIVE_DIR}")

In [ ]:
# ── 0.3  Variables de entorno y parcheo del event loop ───────────────────────
import os, sys, nest_asyncio

# Base de datos en Google Drive
DB_PATH = "/content/drive/MyDrive/InmobiliarioColima/listings.db"
os.environ["DATABASE_URL"] = f"sqlite:///{DB_PATH}"

# Nivel de log: DEBUG | INFO | WARNING
os.environ["LOG_LEVEL"] = "INFO"

# Permite usar await / asyncio.run() dentro de Colab
nest_asyncio.apply()

# Cambia el directorio de trabajo al repo
os.chdir("/content/WebScraperInmobiliario")

print("✅ Entorno configurado.")
print(f"   DATABASE_URL = {os.environ['DATABASE_URL']}")

---
## Sección 1 — Base de datos

In [ ]:
# ── 1.1  Crear tablas (seguro ejecutarlo varias veces) ────────────────────────
from db.database import init_db
init_db()
print("✅ Tablas listas.")

---
## Sección 2 — Scraping

Ajusta las variables `PORTALES` y `OPERACIONES` antes de correr.

In [ ]:
# ── 2.1  Parámetros del scraping ─────────────────────────────────────────────
# Portales disponibles: "inmuebles24", "lamudi", "vivanuncios", "mercadolibre"
PORTALES = ["mercadolibre", "inmuebles24", "lamudi", "vivanuncios"]

# Operaciones disponibles: "venta", "renta"
OPERACIONES = ["venta", "renta"]

print(f"Portales : {PORTALES}")
print(f"Operaciones: {OPERACIONES}")

In [ ]:
# ── 2.2  Ejecutar el scraping ────────────────────────────────────────────────
import asyncio
from pipeline import run_pipeline

summary = asyncio.run(
    run_pipeline(portals=PORTALES, operations=OPERACIONES)
)

# Mostrar resumen
print("\n{'Portal':<20} {'Obtenidos':>10} {'Nuevos':>8} {'Actualizados':>13} {'Errores':>8}")
print("-" * 62)
for portal, stats in summary.items():
    if portal == "_total":
        continue
    print(f"{portal:<20} {stats['fetched']:>10} {stats['inserted']:>8} {stats['updated']:>13} {stats['errors']:>8}")

total = summary.get("_total", {})
print("-" * 62)
print(f"{'TOTAL':<20} {total.get('fetched',0):>10} {total.get('inserted',0):>8} {total.get('updated',0):>13} {total.get('errors',0):>8}")
print(f"\nEstado: {total.get('status', 'desconocido').upper()}")

---
## Sección 3 — Explorar los datos

In [ ]:
# ── 3.1  Cargar listados en un DataFrame ─────────────────────────────────────
import pandas as pd
from sqlalchemy import select, text
from db.database import engine

df = pd.read_sql(
    "SELECT portal, operation, property_type, municipio, colonia, "
    "price, currency, price_per_m2, area_total, area_built, "
    "bedrooms, bathrooms, parking, is_active, first_seen, last_seen "
    "FROM listings WHERE is_active = 1",
    con=engine,
)
print(f"Total de listados activos: {len(df):,}")
df.head(10)

In [ ]:
# ── 3.2  Resumen por portal y operación ──────────────────────────────────────
summary_df = (
    df.groupby(["portal", "operation"])
    .agg(
        total=("price", "count"),
        precio_mediano=("price", "median"),
        precio_min=("price", "min"),
        precio_max=("price", "max"),
    )
    .reset_index()
    .sort_values(["operation", "total"], ascending=[True, False])
)
summary_df[["precio_mediano", "precio_min", "precio_max"]] = (
    summary_df[["precio_mediano", "precio_min", "precio_max"]]
    .applymap(lambda x: f"${x:,.0f}" if pd.notna(x) else "—")
)
summary_df

In [ ]:
# ── 3.3  Distribución por municipio ──────────────────────────────────────────
(
    df[df["operation"] == "venta"]
    .groupby("municipio")["price"]
    .agg(listados="count", precio_mediano="median")
    .sort_values("listados", ascending=False)
    .style.format({"precio_mediano": "${:,.0f}"})
)

---
## Sección 4 — Análisis de mercado

In [ ]:
# ── 4.1  Precio mediano por municipio ────────────────────────────────────────
from analysis.market import median_price_by_municipio

# Cambia operation a "renta" si lo necesitas
result = median_price_by_municipio(operation="venta", property_type=None)
result.style.format({
    "median_price": "${:,.0f}",
    "mean_price":   "${:,.0f}",
    "min_price":    "${:,.0f}",
    "max_price":    "${:,.0f}",
    "std_price":    "${:,.0f}",
})

In [ ]:
# ── 4.2  Precio por m² por municipio ─────────────────────────────────────────
from analysis.market import price_per_m2_stats

result = price_per_m2_stats(operation="venta")
result.style.format({
    "median_ppm2": "${:,.0f}/m²",
    "mean_ppm2":   "${:,.0f}/m²",
    "min_ppm2":    "${:,.0f}/m²",
    "max_ppm2":    "${:,.0f}/m²",
})

In [ ]:
# ── 4.3  Impacto de amenidades en el precio ───────────────────────────────────
from analysis.market import amenity_impact

result = amenity_impact(operation="venta")
if result.empty:
    print("Sin datos suficientes para calcular impacto de amenidades.")
else:
    result.style.format({
        "median_with":    "${:,.0f}",
        "median_without": "${:,.0f}",
        "premium_pct":    "{:+.1f}%",
    }).background_gradient(subset=["premium_pct"], cmap="RdYlGn")

In [ ]:
# ── 4.4  Evolución histórica de precios ──────────────────────────────────────
# Requiere varias ejecuciones del scraper para ver tendencias
from analysis.market import price_history_evolution

# municipio=None → todos los municipios agregados
# freq="W" semanal | "ME" mensual | "D" diario
hist = price_history_evolution(
    municipio=None,   # ← pon un municipio específico si quieres filtrar
    operation="venta",
    freq="W",
)

if hist.empty:
    print("Sin historial de precios aún. Ejecuta el scraper más de una vez.")
else:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(12, 4))
    for mun, grp in hist.groupby("municipio"):
        ax.plot(grp["recorded_at"], grp["median_price"], marker="o", label=mun)
    ax.set_title("Evolución del precio mediano (venta)")
    ax.set_ylabel("Precio MXN")
    ax.legend(bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.show()

---
## Sección 5 — Exportar resultados

In [ ]:
# ── 5.1  Exportar análisis completo a Excel ───────────────────────────────────
from analysis.market import export_summary
from pathlib import Path

EXPORT_DIR = Path("/content/drive/MyDrive/InmobiliarioColima/exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

path_venta = export_summary(operation="venta", output_dir=EXPORT_DIR)
path_renta = export_summary(operation="renta", output_dir=EXPORT_DIR)

print(f"📊 Análisis venta: {path_venta}")
print(f"📊 Análisis renta: {path_renta}")

In [ ]:
# ── 5.2  Exportar listados completos a CSV ────────────────────────────────────
from datetime import datetime
from db.database import engine
import pandas as pd

df_all = pd.read_sql(
    "SELECT * FROM listings WHERE is_active = 1",
    con=engine,
)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = EXPORT_DIR / f"listings_{ts}.csv"
df_all.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"📄 CSV guardado en Drive: {csv_path} ({len(df_all):,} filas)")

In [ ]:
# ── 5.3  Descargar archivos directamente al navegador ─────────────────────────
from google.colab import files

# Descarga el CSV más reciente
files.download(str(csv_path))

# Descarga los Excel de análisis
files.download(str(path_venta))
files.download(str(path_renta))

---
## Sección 6 — Consultas rápidas (opcional)

In [ ]:
# ── 6.1  Top 10 propiedades con mejor precio/m² en Manzanillo ────────────────
import pandas as pd
from db.database import engine

q = """
    SELECT portal, property_type, municipio, colonia,
           price, price_per_m2, area_built, bedrooms, bathrooms, url
    FROM listings
    WHERE is_active = 1
      AND operation = 'venta'
      AND municipio = 'Manzanillo'       -- ← cambia el municipio
      AND price_per_m2 IS NOT NULL
    ORDER BY price_per_m2 ASC
    LIMIT 10
"""
pd.read_sql(q, con=engine)

In [ ]:
# ── 6.2  Historial de ejecuciones del scraper ─────────────────────────────────
import pandas as pd
from db.database import engine

runs = pd.read_sql(
    "SELECT id, started_at, finished_at, portals, total_fetched, "
    "total_inserted, total_updated, total_errors, status "
    "FROM scraping_runs ORDER BY started_at DESC LIMIT 20",
    con=engine,
)
runs